# QC: вхождение ИНН final_df в срез Kedr

View коллег лежит в **DRP**, схема `sbx_edm_dfip`.
Точное имя (права выданы): **`v_detail_dmkb_dkb_dmdsmark`** → `sbx_edm_dfip.v_detail_dmkb_dkb_dmdsmark`.
Три сегмента `Kedr.v_detail_*` — **Impala** (канон витрины, контроль).

Не пишет в `shestopalov_kedr_obshiy_chod_inn_month`.

In [ ]:
import re
from getpass import getpass
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
drp_schema = 'sbx_edm_dfip'
drp_user_login = 'Shestopalov-VYur'
impala_user = 'Shestopalov-VYur'

# Точное имя view, на которое выдали SELECT
DRP_VIEW_NAME = 'v_detail_dmkb_dkb_dmdsmark'
DRP_VIEW_FQ = f'{drp_schema}.{DRP_VIEW_NAME}'
DRP_VIEW_CANDIDATES = [
    f'{drp_schema}.v_detail_dmkb_dkb_dmdsmark',
    f'{drp_schema}._detail_dmkb_dkb_dmdsmark',
    'sbx_da.v_detail_dmkb_dkb_dmdsmark',
    'sbx_da._detail_dmkb_dkb_dmdsmark',
]

view_name_needles = (
    'kedr', 'chod', 'nbi', 'shestopalov', 'acquiring', 'ekvair',
    'dmk', 'dmsb', 'dkb', 'obshiy', 'dmdsmark', 'dmkb_dkb',
    'v_detail_dmkb_dkb_dmdsmark', '_detail_dmkb_dkb_dmdsmark',
)

final_df_candidates = [
    DATA_DIR / 'final_df_period_2026_01_2026_08_final_script_2.csv',
    DATA_DIR / 'final_df_period_2026_01_2026_07_final_script_2.csv',
    DATA_DIR / 'final_df_period_2026_01_2026_07_mpos.csv',
]

kedr_sources = {
    'dmkb': 'Kedr.v_detail_dmkb',
    'dmsb': 'Kedr.v_detail_dmsb',
    'dkb': 'Kedr.v_detail_dkb',
}
chunk_size = 800
mem_limit = '8g'

output_xlsx = DATA_DIR / 'qc_kedr_inn_coverage.xlsx'
letter_md_path = DATA_DIR / 'qc_kedr_inn_coverage_letter.md'

print('DRP schema:', drp_schema)
print('DRP view:', DRP_VIEW_FQ)
print('DATA_DIR', DATA_DIR, 'exists=', DATA_DIR.exists())

In [ ]:
def normalize_inn(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s


def in_sql_list(values):
    clean = [str(v).strip() for v in values if str(v).strip()]
    if not clean:
        return "''"
    return ', '.join("'" + v.replace("'", "''") + "'" for v in clean)


def chunked(values, size):
    for i in range(0, len(values), size):
        yield values[i:i + size]


def fetch_drp(sql, label='DRP'):
    print(f'[{pd.Timestamp.now().strftime("%H:%M:%S")}] {label}')
    try:
        with drp:
            out = drp.fetch(sql)
    except Exception as exc:
        msg = f'{type(exc).__name__} {exc}'.lower()
        if 'ldap' in msg or 'authentication failed' in msg:
            raise RuntimeError(
                'DRP LDAP authentication failed. Логин/пароль не приняты. '
                'Перезапустите ячейку connect (секция 1) с Shestopalov-VYur и рабочим паролем. '
                f'Исходная ошибка: {exc}'
            ) from exc
        raise
    return pd.DataFrame() if out is None else out


def fetch_imp(sql, label='Impala'):
    print(f'[{pd.Timestamp.now().strftime("%H:%M:%S")}] {label}')
    with imp:
        try:
            imp.execute(f'set MEM_LIMIT={mem_limit}')
        except Exception:
            pass
        out = imp.fetch(sql)
    return pd.DataFrame() if out is None else out

## 1. DRP: connect и view в `sbx_edm_dfip`

In [ ]:
FORCE_DRP_RECONNECT = True

def _drp_is_ldap_error(exc):
    msg = f'{type(exc).__name__} {exc}'.lower()
    return 'ldap' in msg or 'authentication failed' in msg or 'password' in msg


def ping_drp():
    with drp:
        out = drp.fetch('SELECT current_user AS current_user, 1 AS ok')
    return pd.DataFrame() if out is None else out


print('NOTE: строка «DRP connected» без таблицы current_user — это старая ячейка, логина нет.')
drp = None
print('Логин:', drp_user_login, '| тот же пароль, что для sbx_da витрины')
drp_user = input('DRP user: ').strip() or drp_user_login
try:
    drp_password = getpass('DRP password: ')
except Exception:
    drp_password = input('DRP password (getpass недоступен, ввод будет виден): ')
print('user=', drp_user, '| password length=', len(str(drp_password or '')))
if not str(drp_password).strip():
    raise RuntimeError('Пароль пустой — Jupyter не принял getpass. Запусти ячейку ещё раз.')

drp = connect(
    to='DRP',
    user_params={'user_name': drp_user, 'password': drp_password},
)
if hasattr(drp, '_init_connection'):
    try:
        drp._init_connection()
    except Exception as exc:
        print('_init_connection:', type(exc).__name__, exc)

try:
    ping = ping_drp()
    display(ping)
    print('DRP AUTH OK as', drp_user)
except Exception as exc:
    if _drp_is_ldap_error(exc):
        raise RuntimeError(
            f'LDAP отверг {drp_user!r} (длина пароля {len(str(drp_password))}). '
            'Строка «DRP connected» ничего не значит. Нужен пароль Greenplum/DRP, не keytab Impala. '
            f'{exc}'
        ) from exc
    raise

ctrl = fetch_drp("SELECT COUNT(*) AS n FROM information_schema.tables WHERE table_schema = 'sbx_da'", 'control sbx_da')
display(ctrl)
print('Если control sbx_da > 0 — логин живой, можно читать view.')

In [ ]:
schema_err = None
drp_objects = pd.DataFrame()
try:
    drp_objects = fetch_drp(f'''
    SELECT
      table_schema,
      table_name,
      table_type
    FROM information_schema.tables
    WHERE lower(table_schema) = lower('{drp_schema}')
    ORDER BY table_type, table_name
    ''', f'objects in {drp_schema}')
except Exception as exc:
    schema_err = f'{type(exc).__name__}: {exc}'
    print('ERROR schema listing:', schema_err)

print('=== Объекты', drp_schema, '===')
if drp_objects is None or drp_objects.empty:
    print('Пусто или нет USAGE на схему. schema_err=', schema_err)
else:
    display(drp_objects)
    if 'table_type' in drp_objects.columns:
        print('types:', drp_objects['table_type'].value_counts().to_dict())

In [ ]:
# Контроль: схема существует? свой sbx_da виден?
try:
    schemata = fetch_drp(f"""
    SELECT schema_name
    FROM information_schema.schemata
    WHERE lower(schema_name) LIKE 'sbx_ed%'
       OR lower(schema_name) IN (lower('{drp_schema}'), 'sbx_da')
    ORDER BY 1
    """, 'schemata sbx_ed* / sbx_da')
    print('=== Схемы sbx_ed* и sbx_da ===')
    display(schemata)
except Exception as exc:
    print('schemata failed:', type(exc).__name__, exc)
    schemata = pd.DataFrame()

try:
    sbx_da_n = fetch_drp("""
    SELECT COUNT(*) AS n
    FROM information_schema.tables
    WHERE lower(table_schema) = 'sbx_da'
    """, 'count tables in sbx_da')
    print('=== Сколько объектов в sbx_da (контроль каталога) ===')
    display(sbx_da_n)
except Exception as exc:
    print('sbx_da count failed:', type(exc).__name__, exc)

print('ClickHouse JDBC WARNING — шум log4j, не значит что вы в ClickHouse.')
if 'schema_err' in globals() and schema_err and ('ldap' in str(schema_err).lower() or 'authentication' in str(schema_err).lower()):
    print('VERDICT: LDAP не пустил в DRP. View и USAGE тут ни при чём. Перелогиньтесь в секции 1.')
elif schemata is None or schemata.empty:
    print('VERDICT: схемы sbx_edm_dfip в каталоге не видно. Имя неверное или нет USAGE.')
elif not schemata['schema_name'].astype(str).str.lower().eq(drp_schema.lower()).any():
    print('VERDICT: sbx_edm_dfip нет в списке. Смотрите соседние sbx_ed*.')
else:
    print('VERDICT: схема есть, объектов нет — нет USAGE или view ещё не создали.')


In [ ]:
drp_views = pd.DataFrame()
try:
    drp_views = fetch_drp(f'''
    SELECT table_schema, table_name
    FROM information_schema.views
    WHERE lower(table_schema) = lower('{drp_schema}')
    ORDER BY table_name
    ''', f'views in {drp_schema}')
except Exception as exc:
    print('information_schema.views failed:', type(exc).__name__, exc)
    try:
        drp_views = fetch_drp(f'''
        SELECT
          n.nspname AS table_schema,
          c.relname AS table_name
        FROM pg_class c
        JOIN pg_namespace n ON n.oid = c.relnamespace
        WHERE lower(n.nspname) = lower('{drp_schema}')
          AND c.relkind IN ('v', 'm')
        ORDER BY 2
        ''', 'pg_class views')
    except Exception as exc2:
        print('pg_class failed:', type(exc2).__name__, exc2)

kedr_like = pd.DataFrame()
if drp_views is None or drp_views.empty:
    print('VIEW не видно. Если objects выше не пустой — ищите table_type VIEW там.')
    if drp_objects is not None and len(drp_objects) and 'table_type' in drp_objects.columns:
        kedr_like = drp_objects[
            drp_objects['table_type'].astype(str).str.upper().str.contains('VIEW')
        ].copy()
else:
    display(drp_views)
    kedr_like = drp_views[
        drp_views['table_name'].astype(str).str.lower().apply(
            lambda s: any(n in s for n in view_name_needles)
        )
    ].copy()

print('=== Похоже на Kedr / эквайринг ===')
display(kedr_like if kedr_like is not None and len(kedr_like) else '(нет совпадений по имени)')
if kedr_like is not None and len(kedr_like):
    for _, r in kedr_like.iterrows():
        print(f"  {r.get('table_schema', drp_schema)}.{r.get('table_name')}")

In [ ]:
try:
    who = fetch_drp('''
    SELECT current_user AS current_user, session_user AS session_user
    ''', 'whoami')
    print('=== Кто в сессии DRP ===')
    display(who)
except Exception as exc:
    print('whoami failed:', type(exc).__name__, exc)
    who = pd.DataFrame()

try:
    schema_priv = fetch_drp(f'''
    SELECT
      has_schema_privilege(current_user, '{drp_schema}', 'USAGE') AS usage_ok,
      has_schema_privilege(current_user, '{drp_schema}', 'CREATE') AS create_ok
    ''', 'schema privilege')
    print('=== USAGE на', drp_schema, '===')
    display(schema_priv)
except Exception as exc:
    print('schema privilege failed:', type(exc).__name__, exc)
    schema_priv = pd.DataFrame()

try:
    my_roles = fetch_drp('''
    SELECT r.rolname AS role_name
    FROM pg_auth_members m
    JOIN pg_roles r ON r.oid = m.roleid
    JOIN pg_roles u ON u.oid = m.member
    WHERE lower(u.rolname) = lower(current_user)
    ORDER BY 1
    ''', 'my roles')
    print('=== Роли current_user ===')
    display(my_roles)
except Exception as exc:
    print('roles failed:', type(exc).__name__, exc)
    my_roles = pd.DataFrame()

try:
    grants_login = fetch_drp(f'''
    SELECT table_schema, table_name, grantee, privilege_type
    FROM information_schema.role_table_grants
    WHERE lower(table_schema) = lower('{drp_schema}')
      AND (
            lower(CAST(grantee AS TEXT)) = lower('{drp_user}')
         OR lower(CAST(grantee AS TEXT)) = lower('{drp_user_login}')
         OR lower(CAST(grantee AS TEXT)) = lower(current_user)
      )
    ORDER BY table_name, privilege_type
    ''', 'grants on login')
    print('=== GRANT на логине / current_user ===')
    display(grants_login)
    if grants_login is None or grants_login.empty:
        print('На логине пусто — нормально, если права выданы роли.')
except Exception as exc:
    print('grants on login failed:', type(exc).__name__, exc)
    grants_login = pd.DataFrame()

try:
    grants_all = fetch_drp(f'''
    SELECT table_schema, table_name, grantee, privilege_type
    FROM information_schema.role_table_grants
    WHERE lower(table_schema) = lower('{drp_schema}')
    ORDER BY table_name, grantee, privilege_type
    LIMIT 300
    ''', 'all grants in schema')
    print('=== Все GRANT в схеме (до 300) ===')
    display(grants_all)
except Exception as exc:
    print('all grants failed:', type(exc).__name__, exc)
    grants_all = pd.DataFrame()

n_obj = 0 if drp_objects is None else len(drp_objects)
n_view = 0 if ('drp_views' not in globals() or drp_views is None) else len(drp_views)
print(f'objects={n_obj} views={n_view}')
if n_obj == 0 and n_view == 0:
    print('Схема пустая для вас: нет USAGE или view ещё не создали.')
    print('Письмо: USAGE ON SCHEMA sbx_edm_dfip + SELECT на их view.')
else:
    print('Объекты видны — SELECT проверяйте ячейкой found_view_fq, не каталогом GRANT.')

View зафиксирован: `sbx_edm_dfip.v_detail_dmkb_dkb_dmdsmark`. Если SELECT не пройдёт — тетрадка переберёт запасные имена.

In [ ]:
found_view_fq = DRP_VIEW_FQ
print('found_view_fq (задан) =', repr(found_view_fq))

view_probe = pd.DataFrame()
view_cols = pd.DataFrame()
view_access_ok = False
last_err = None
tried = []
for cand in DRP_VIEW_CANDIDATES:
    sch, nam = cand.split('.', 1)
    tried.append(cand)
    try:
        view_cols = fetch_drp(f'''
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE lower(table_schema) = lower('{sch}')
          AND lower(table_name) = lower('{nam}')
        ORDER BY ordinal_position
        ''', f'columns {cand}')
        view_probe = fetch_drp(f'SELECT * FROM {cand} LIMIT 5', f'SELECT {cand} LIMIT 5')
        found_view_fq = cand
        view_access_ok = True
        print('OK:', found_view_fq)
        print('=== Колонки ===')
        display(view_cols)
        print('=== 5 строк (доступ есть) ===')
        display(view_probe)
        break
    except Exception as exc:
        last_err = exc
        print('FAIL', cand, type(exc).__name__, exc)

if not view_access_ok:
    print('SELECT не прошёл ни по одному имени. tried=', tried)
    if last_err is not None:
        print(type(last_err).__name__, last_err)

## 2. ИНН из final_df

In [ ]:
final_df_csv_path = next((p for p in final_df_candidates if p.exists()), None)
if final_df_csv_path is None:
    raise RuntimeError(
        'Нет CSV final_df в DATA_DIR. Ожидали одно из: '
        + ', '.join(p.name for p in final_df_candidates)
    )

print('final_df:', final_df_csv_path)
final_src_df = pd.read_csv(final_df_csv_path, dtype=str, low_memory=False)
if 'inn' not in final_src_df.columns:
    raise RuntimeError('Нет колонки inn. cols=' + str(list(final_src_df.columns)[:40]))

work = final_src_df.copy()
work['inn'] = work['inn'].map(normalize_inn)
if 'report_month' in work.columns:
    work['report_month'] = work['report_month'].astype(str).str[:7]
else:
    work['report_month'] = ''

inn_month = (
    work.dropna(subset=['inn'])
    .groupby(['inn', 'report_month'], as_index=False)
    .size()
    .rename(columns={'size': 'agr_rows'})
)
inn_values = sorted(inn_month['inn'].unique().tolist())
months_in_csv = sorted(x for x in inn_month['report_month'].unique() if x and x != 'nan')
print(f'rows={len(work):,} | unique INN={len(inn_values):,} | months={months_in_csv}')
display(inn_month.head(10))

## 3. Coverage: Kedr.v_detail_* (Impala)

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': impala_user},
)
imp._init_connection()
print('Impala connected')

In [ ]:
def month_to_yearmm(ym):
    parts = str(ym).split('-')
    if len(parts) < 2:
        return None
    return int(parts[0]) * 100 + int(parts[1])


def fetch_kedr_inns(inn_scope, yyyymm, src_name):
    fq = kedr_sources[src_name]
    inn_sql = in_sql_list(inn_scope)
    sql = f'''
    SELECT CAST(inn AS STRING) AS inn
    FROM {fq}
    WHERE yearmm = {yyyymm}
      AND CAST(inn AS STRING) IN ({inn_sql})
    GROUP BY CAST(inn AS STRING)
    '''
    try:
        df = fetch_imp(sql, f'{src_name} {yyyymm} n={len(inn_scope)}')
    except Exception as exc:
        print('FAIL', src_name, yyyymm, type(exc).__name__, exc)
        return set(), str(exc)
    if df is None or df.empty:
        return set(), None
    return set(df['inn'].map(normalize_inn).dropna().tolist()), None


target_months = [m for m in months_in_csv if month_to_yearmm(m)]
hit_rows = []
src_errors = []

for ym in target_months:
    yyyymm = month_to_yearmm(ym)
    inns_m = inn_month.loc[inn_month['report_month'] == ym, 'inn'].tolist()
    found = {k: set() for k in kedr_sources}
    for src in kedr_sources:
        acc = set()
        for chunk in chunked(inns_m, chunk_size):
            part, err = fetch_kedr_inns(chunk, yyyymm, src)
            if err:
                src_errors.append({'report_month': ym, 'src': src, 'error': err})
            acc |= part
        found[src] = acc
    union = found['dmkb'] | found['dmsb'] | found['dkb']
    for inn in inns_m:
        hit_rows.append({
            'inn': inn,
            'report_month': ym,
            'yearmm': yyyymm,
            'in_dmkb': int(inn in found['dmkb']),
            'in_dmsb': int(inn in found['dmsb']),
            'in_dkb': int(inn in found['dkb']),
            'in_any_zo': int(inn in union),
            'zo_cnt': int(inn in found['dmkb']) + int(inn in found['dmsb']) + int(inn in found['dkb']),
        })

hits = pd.DataFrame(hit_rows)
print('src_errors:', len(src_errors))
if src_errors:
    display(pd.DataFrame(src_errors))
display(hits.head(10))

In [ ]:
if hits.empty:
    raise RuntimeError('Нет hits — проверьте Impala / месяцы CSV')

by_month = hits.groupby('report_month', as_index=False).agg(
    inns_final_df=('inn', 'nunique'),
    inns_in_any_zo=('in_any_zo', 'sum'),
    inns_dmkb=('in_dmkb', 'sum'),
    inns_dmsb=('in_dmsb', 'sum'),
    inns_dkb=('in_dkb', 'sum'),
    inns_only_one_zo=('zo_cnt', lambda s: int((s == 1).sum())),
)
by_month['coverage_pct'] = np.where(
    by_month['inns_final_df'] > 0,
    100.0 * by_month['inns_in_any_zo'] / by_month['inns_final_df'],
    np.nan,
)

inn_any = hits.groupby('inn', as_index=False).agg(
    months=('report_month', lambda s: ','.join(sorted(set(s)))),
    months_in_kedr=('in_any_zo', 'sum'),
    in_any_zo=('in_any_zo', 'max'),
    in_dmkb=('in_dmkb', 'max'),
    in_dmsb=('in_dmsb', 'max'),
    in_dkb=('in_dkb', 'max'),
)
missing_inn = inn_any.loc[inn_any['in_any_zo'] == 0].copy()
missing_month = hits.loc[hits['in_any_zo'] == 0].copy()

print('=== Покрытие по месяцам ===')
display(by_month)
print(
    f"unique INN={inn_any['inn'].nunique():,} | в Kedr ЗО={int(inn_any['in_any_zo'].sum()):,} | "
    f"нет={len(missing_inn):,}"
)
display(missing_inn.head(20))

## 4. Те же ИНН против DRP view

In [ ]:
view_hits = pd.DataFrame()
view_missing = pd.DataFrame()
if not str(found_view_fq).strip() or not view_access_ok:
    print('SKIP сверка с DRP view (нет имени или нет SELECT).')
else:
    cols_l = set()
    if view_cols is not None and len(view_cols) and 'column_name' in view_cols.columns:
        cols_l = set(view_cols['column_name'].astype(str).str.lower())
    inn_col = next((c for c in ('inn', 'inn_id', 'c_inn', 'inn_org', 'client_inn', 'ul_inn') if c in cols_l), None)
    prod_col = next((c for c in ('product_name', 'prod_name', 'product', 'item', 'c_name') if c in cols_l), None)
    print('mapped inn=', inn_col, 'product=', prod_col)
    if inn_col is None:
        print('В view нет колонки inn.')
    else:
        extra_prod = ''
        if prod_col:
            extra_prod = f"AND CAST({prod_col} AS TEXT) ILIKE '%эквайринг%'"
        rows = []
        for chunk in chunked(inn_values, chunk_size):
            inn_sql = in_sql_list(chunk)
            sql = f'''
            SELECT DISTINCT CAST({inn_col} AS TEXT) AS inn
            FROM {found_view_fq}
            WHERE CAST({inn_col} AS TEXT) IN ({inn_sql})
            {extra_prod}
            '''
            part = fetch_drp(sql, f'DRP view inns chunk {len(chunk)}')
            if part is not None and len(part):
                rows.append(part)
        found_set = set()
        if rows:
            found_set = set(pd.concat(rows, ignore_index=True)['inn'].map(normalize_inn).dropna())
        view_hits = pd.DataFrame({
            'inn': inn_values,
            'in_drp_view': [int(x in found_set) for x in inn_values],
        })
        view_missing = view_hits.loc[view_hits['in_drp_view'] == 0].copy()
        print(f'INN в DRP view: {int(view_hits["in_drp_view"].sum())} / {len(inn_values)}')
        display(view_missing.head(20))

## 5. Вердикт и черновик

In [ ]:
n_inn = int(inn_any['inn'].nunique()) if 'inn_any' in globals() and len(inn_any) else 0
n_ok = int(inn_any['in_any_zo'].sum()) if n_inn else 0
n_miss = int(len(missing_inn)) if 'missing_inn' in globals() else 0
n_dmsb = int(inn_any['in_dmsb'].sum()) if n_inn and 'in_dmsb' in inn_any.columns else 0

if schema_err:
    view_status = f'нет доступа к схеме {drp_schema}: {schema_err}'
elif found_view_fq and view_access_ok:
    view_status = f'SELECT ок (DRP): {found_view_fq}'
elif found_view_fq:
    view_status = f'view указан, SELECT не прошёл: {found_view_fq}'
else:
    view_status = f'view в {drp_schema} не выбран; сверка по Kedr.v_detail_*'

if n_miss == 0 and n_inn:
    verdict = 'все ИНН final_df есть хотя бы в одном ЗО Kedr'
elif n_dmsb == 0 and n_ok:
    verdict = 'дыра ДМСБ: в dmsb нет ни одного ИНН витрины'
elif n_miss:
    verdict = f'дыра среза: {n_miss} ИНН витрины нет ни в dmkb/dmsb/dkb'
else:
    verdict = 'нет данных сверки'

print('VERDICT:', verdict)
print('view:', view_status)

lines = []
lines.append('Кристина, добрый день.')
lines.append('')
lines.append(
    'Проверяем вхождение ИНН витрины торгового эквайринга (final_df) '
    'в срез Kedr по трём ЗО: ДМКБ, ДМСБ, ДКБ.'
)
lines.append(f'View DRP {drp_schema}: {view_status}.')
lines.append('')
lines.append(f'ИНН в final_df: {n_inn}. Есть хотя бы в одном v_detail_*: {n_ok}. Нет ни в одном: {n_miss}.')
if 'by_month' in globals() and by_month is not None and len(by_month):
    for _, r in by_month.iterrows():
        lines.append(
            f"  {r['report_month']}: {int(r['inns_final_df'])} / {int(r['inns_in_any_zo'])} / "
            f"{r['coverage_pct']:.1f}% / dmsb={int(r['inns_dmsb'])}"
        )
if n_dmsb == 0:
    lines.append('В dmsb (ДМСБ) совпадений нет. Нужны все три ЗО.')
if n_miss:
    lines.append(f'Отсутствующие ИНН — qc_kedr_inn_coverage.xlsx, missing_inn ({n_miss} шт.).')
if 'view_missing' in globals() and view_missing is not None and len(view_missing):
    lines.append(f'В DRP view нет {len(view_missing)} ИНН витрины.')
lines.append('Просьба: точное имя view и фильтры zo + product.')
letter = '\n'.join(lines)
print('=== Черновик ===')
print(letter)

try:
    letter_md_path.write_text(letter + '\n', encoding='utf-8')
    print('Saved', letter_md_path)
except Exception as exc:
    print('Save letter failed:', type(exc).__name__, exc)

try:
    with pd.ExcelWriter(output_xlsx, engine='openpyxl') as w:
        if drp_objects is not None and len(drp_objects):
            drp_objects.to_excel(w, sheet_name='drp_objects', index=False)
        if 'drp_views' in globals() and drp_views is not None and len(drp_views):
            drp_views.to_excel(w, sheet_name='drp_views', index=False)
        if 'by_month' in globals() and by_month is not None:
            by_month.to_excel(w, sheet_name='coverage_month', index=False)
        if 'missing_inn' in globals() and missing_inn is not None:
            missing_inn.to_excel(w, sheet_name='missing_inn', index=False)
        if 'missing_month' in globals() and missing_month is not None:
            missing_month.to_excel(w, sheet_name='missing_inn_month', index=False)
        if view_hits is not None and len(view_hits):
            view_hits.to_excel(w, sheet_name='drp_view_hits', index=False)
        pd.DataFrame([{
            'verdict': verdict,
            'view_status': view_status,
            'found_view_fq': found_view_fq,
            'n_inn': n_inn,
            'n_ok': n_ok,
            'n_miss': n_miss,
            'n_dmsb': n_dmsb,
        }]).to_excel(w, sheet_name='verdict', index=False)
    print('Saved', output_xlsx)
except Exception as exc:
    print('Save xlsx failed:', type(exc).__name__, exc)